# Clean up messy data

**The job.** Take a table with the usual problems. Fix what can be fixed. Record
what was done. Flag what could not be.

The record is the deliverable. A cleaned file with no report is a file you
cannot trust, because you cannot see what was changed.

**In:** a messy CSV.
**Out:** a clean CSV plus a report of every change.
**Files:** messy.csv, clean.csv, cleaning-report.json.

In [1]:
try:
    import browsergraph  # noqa: F401
except ImportError:
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import json, pathlib
from dataclasses import replace

from browsergraph import execute, viz
from browsergraph.compile import compile_route
from browsergraph.manifest import NodeManifest, PortSpec
from browsergraph.workbench import Edge, NodeCandidate, StageDefinition, WorkbenchDefinition

# A fresh folder each run. Left-over files from a previous run make the "what
# did this produce" list a lie, and that list is half the point here.
import shutil
WORK = pathlib.Path("work")
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

def node(node_id, capability, takes, gives, **kw):
    """Describe one node. Ports are (name, type) pairs."""
    return NodeManifest(
        id=node_id, kind="function", description=f"{capability} via {node_id}",
        capabilities=(capability,),
        inputs=tuple(PortSpec(n, t) for n, t in takes),
        outputs=tuple(PortSpec(n, t) for n, t in gives), **kw)

def stage(sid, name, takes, gives, capability, candidates):
    """Describe one step of the job, and what could do it."""
    return StageDefinition(
        id=sid, name=name, required_capabilities=(capability,),
        inputs=tuple(PortSpec(n, t) for n, t in takes),
        outputs=tuple(PortSpec(n, t) for n, t in gives),
        success=f"{name} produced its declared output",
        candidates=tuple(candidates))

def build(title, task, stages, nodes, edges=()):
    """Put it together and check it before anything runs."""
    bench = WorkbenchDefinition(
        title=title, task=task, stages=tuple(stages), nodes=tuple(nodes),
        edges=tuple(edges),
        candidates=tuple(NodeCandidate(id=n.id, node_id=n.id) for n in nodes))
    problems = bench.validate()
    print("problems:", problems if problems else "none")
    return bench

print("ready")

ready


## The input

Twelve rows with six problems in them: duplicate rows, padded whitespace,
mixed case in a category, numbers stored as text with symbols, a missing value,
and one outlier that is clearly a data entry slip.

In [2]:
MESSY = """id,city,category,amount,joined
1, London ,Retail,"$1,200.00",2026-01-05
2,Leeds,retail,$980.50,2026-01-09
3, Leeds,RETAIL,$1;050,2026-01-11
4,Bristol,Wholesale,$640.00,2026-02-01
2,Leeds,retail,$980.50,2026-01-09
5,london,Retail,,2026-02-14
6,Bristol ,wholesale,$720.25,2026-02-20
7,Leeds,Retail,$99999999.00,2026-03-02
8,Bristol,Wholesale,$540.10,2026-03-05
9, London,retail,$1,340.00,2026-03-11
10,Leeds,Wholesale,$610.00,2026-03-18
9, London,retail,$1,340.00,2026-03-11"""

source = WORK / "messy.csv"
source.write_text(MESSY)
print(MESSY[:260], "...")
print(f"\n{len(MESSY.splitlines()) - 1} data rows")

id,city,category,amount,joined
1, London ,Retail,"$1,200.00",2026-01-05
2,Leeds,retail,$980.50,2026-01-09
3, Leeds,RETAIL,$1;050,2026-01-11
4,Bristol,Wholesale,$640.00,2026-02-01
2,Leeds,retail,$980.50,2026-01-09
5,london,Retail,,2026-02-14
6,Bristol ,wholesal ...

12 data rows


## The steps

Load, look at what is wrong, then fix in two independent passes — text tidying
and number parsing — and bring them back together before the final check.

Two passes because they are separate concerns. Trimming whitespace has nothing
to do with parsing currency, they can be reviewed separately, and either can be
swapped without touching the other.

In [3]:
nodes = [
    node("load.csv",    "data.read",    [],                     [("out", "Rows")]),
    node("profile.rows","data.profile", [("in", "Rows")],       [("out", "Profile")]),
    node("fix.text",    "clean.text",   [("in", "Rows")],       [("out", "Rows")]),
    node("fix.numbers", "clean.numbers",[("in", "Rows")],       [("out", "Rows"), ("repairs", "Notes")]),
    node("merge.fixes", "clean.merge",  [("text", "Rows"), ("numbers", "Rows")], [("out", "Rows")]),
    node("final.check", "validate",     [("in", "Rows"), ("profile", "Profile"), ("repairs", "Notes")], [("rows", "Rows"), ("report", "Report")]),
    node("write.out",   "write",        [("rows", "Rows"), ("report", "Report")], [("out", "Receipt")],
         effects=("file.write",)),
]

stages = [
    stage("load",    "Load the CSV",    [],                [("out", "Rows")],    "data.read",     ["load.csv"]),
    stage("profile", "See what is wrong",[("in", "Rows")], [("out", "Profile")], "data.profile",  ["profile.rows"]),
    stage("text",    "Tidy the text",   [("in", "Rows")],  [("out", "Rows")],    "clean.text",    ["fix.text"]),
    stage("numbers", "Parse the numbers",[("in", "Rows")], [("out", "Rows"), ("repairs", "Notes")], "clean.numbers", ["fix.numbers"]),
    stage("merge",   "Merge both fixes",[("text", "Rows"), ("numbers", "Rows")], [("out", "Rows")], "clean.merge", ["merge.fixes"]),
    stage("check",   "Final check",     [("in", "Rows"), ("profile", "Profile"), ("repairs", "Notes")], [("rows", "Rows"), ("report", "Report")], "validate", ["final.check"]),
    stage("write",   "Write results",   [("rows", "Rows"), ("report", "Report")], [("out", "Receipt")], "write", ["write.out"]),
]

edges = [Edge("load", "profile"), Edge("load", "text"), Edge("load", "numbers"),
         Edge("text", "merge", to_port="text"),
         Edge("numbers", "merge", from_port="out", to_port="numbers"),
         Edge("numbers", "check", from_port="repairs", to_port="repairs"),
         Edge("merge", "check", to_port="in"),
         Edge("profile", "check", to_port="profile"),
         Edge("check", "write", from_port="rows", to_port="rows"),
         Edge("check", "write", from_port="report", to_port="report")]

bench = build("Clean up a messy table",
              "Fix what can be fixed, and write down everything that was changed.",
              stages, nodes, edges)
print("layers:", bench.layers())

problems: none
layers: [['load'], ['profile', 'text', 'numbers'], ['merge'], ['check'], ['write']]


In [4]:
viz.dag(bench)

Figure(svg='<svg viewBox="0 0 1100 390" width="1100" height="390" style="max-width:none" role="img"><defs><marker id="bg98292286-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Load the CSV</text><text x="69" y="190.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="363.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1 · 3 parallel</text><g><rect x="270" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="279" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">See what is wrong</text><text x="279" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><g><rect x="270" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="279" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Tidy the text</text><text x="279" y="190.0" font-size="9.5" fill="#68737f">1 candidate</text></g><g><rect x="270" y="238.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="279" y="257.0" font-size="11.5" font-weight="700" fill="#22303f">Parse the numbers</text><text x="279" y="272.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="573.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2</text><g><rect x="480" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="489" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Merge both fixes</text><text x="489" y="190.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="783.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 3</text><g><rect x="690" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="699" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Final check</text><text x="699" y="190.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="993.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 4</text><g><rect x="900" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="909" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Write results</text><text x="909" y="190.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,182.0 C258.0,182.0 258.0,100.0 270,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg98292286-arrow)"/><path d="M246,182.0 C258.0,182.0 258.0,182.0 270,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg98292286-arrow)"/><path d="M246,182.0 C258.0,182.0 258.0,264.0 270,264.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg98292286-arrow)"/><path d="M456,182.0 C468.0,182.0 468.0,182.0 480,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg98292286-arrow)"/><text x="468.0" y="177.0" text-anchor="middle" font-size="9" fill="#68737f">text</text><path d="M456,264.0 C468.0,264.0 468.0,182.0 480,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg98292286-arrow)"/><text x="468.0" y="218.0" text-anchor="middle" font-size="9" fill="#68737f">numbers</text><path d="M456,264.0 C573.0,264.0 573.0,182.0 690,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg98292286-arrow)"/><text x="573.0" y="218.0" text-anchor="middle" font-size="9" fill="#68737f">repairs</text><path d="M666,182.0 C678.0,182.0 678.0,182.0 690,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4

In [5]:
import csv, io, re
from collections import Counter

def load_csv():
    """`restkey` catches rows with too many fields instead of losing them.

    Row 9 is `$1,340.00` with no quotes around it. The comma inside the number
    is also the column separator, so that row arrives with six fields where the
    header has five. Without `restkey` the spare piece is silently dropped and
    the row looks fine while holding the wrong amount and the wrong date.
    """
    return list(csv.DictReader(io.StringIO(source.read_text()),
                               restkey="_extra", restval=""))

def profile_rows(**kw):
    rows = kw["in"]
    plain = [{k: v for k, v in r.items() if k != "_extra"} for r in rows]
    seen = Counter(tuple(sorted(r.items())) for r in plain)
    return {
        "rows": len(rows),
        "duplicate_rows": sum(c - 1 for c in seen.values() if c > 1),
        "split_by_comma": sum(1 for r in rows if r.get("_extra")),
        "padded_values": sum(1 for r in plain for v in r.values() if v != v.strip()),
        "blank_values": sum(1 for r in plain for v in r.values() if not v.strip()),
        "city_spellings": sorted({r["city"].strip() for r in rows}),
        "category_spellings": sorted({r["category"].strip() for r in rows}),
    }

def fix_text(**kw):
    """Trim spaces and settle on one spelling per city and category."""
    out = []
    for row in kw["in"]:
        fixed = {k: (v.strip() if isinstance(v, str) else v)
                 for k, v in row.items() if k != "_extra"}
        fixed["city"] = fixed["city"].title()
        fixed["category"] = fixed["category"].title()
        out.append(fixed)
    return out

def fix_numbers(**kw):
    """Currency text into a number, repairing the rows the comma broke.

    Two repairs, both worth naming:

    * `$1` + `340.00` + a spare date is one amount that got cut in half by its
      own thousands separator. Glue it back and shift the date along.
    * `$1;050` is a typo for `$1,050`. The semicolon is next to the comma.
    """
    out, repairs = [], []
    for row in kw["in"]:
        row = dict(row)
        spare = row.pop("_extra", "") or ""
        if spare:
            joined = spare[0] if isinstance(spare, list) else spare
            repairs.append(f"id {row['id'].strip()}: amount was split by its own comma")
            row["amount"] = f"{row['amount']},{row['joined']}"
            row["joined"] = joined
        raw = (row.get("amount") or "").strip()
        digits = re.sub(r"[^0-9.]", "", raw.replace(";", ""))
        try:
            amount = float(digits) if digits else None
        except ValueError:
            amount = None
        out.append(dict(row, amount=amount))
    return {"out": out, "repairs": repairs}

def merge_fixes(**kw):
    """Text fixes and number fixes, side by side, row for row.

    Both passes started from the same rows in the same order, so lining them up
    by position is safe. If either pass ever dropped a row that would stop being
    true, which is why neither of them does.
    """
    return [dict(text_row, amount=number_row["amount"], joined=number_row["joined"])
            for text_row, number_row in zip(kw["text"], kw["numbers"])]

def final_check(**kw):
    """Drop duplicates, flag missing and absurd values. Record all of it."""
    rows, seen = [], set()
    notes = [{"id": r.split(":")[0].replace("id ", ""), "action": "repaired",
              "why": r.split(": ", 1)[1]} for r in kw["repairs"]]
    amounts = [r["amount"] for r in kw["in"] if r["amount"] is not None]
    ceiling = (sorted(amounts)[len(amounts) // 2]) * 20 if amounts else float("inf")

    for row in kw["in"]:
        key = (row["id"], row["joined"])
        if key in seen:
            notes.append({"id": row["id"], "action": "dropped", "why": "duplicate row"})
            continue
        seen.add(key)
        if row["amount"] is None:
            notes.append({"id": row["id"], "action": "flagged", "why": "no amount"})
            rows.append(dict(row, flag="missing amount")); continue
        if row["amount"] > ceiling:
            notes.append({"id": row["id"], "action": "flagged",
                          "why": f"amount {row['amount']:,.0f} is far above the rest"})
            rows.append(dict(row, flag="suspicious amount")); continue
        rows.append(dict(row, flag=""))

    report = {"before": kw["profile"], "changes": notes,
              "after": {"rows": len(rows),
                        "flagged": sum(1 for r in rows if r["flag"])}}
    return {"rows": rows, "report": report}

def write_out(workspace, **kw):
    path = workspace / "clean.csv"
    fields = ["id", "city", "category", "amount", "joined", "flag"]
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for row in kw["rows"]:
            writer.writerow({f: row.get(f, "") for f in fields})
    (workspace / "cleaning-report.json").write_text(json.dumps(kw["report"], indent=2))
    return {"rows_written": len(kw["rows"])}

runtime = execute.Runtime({
    "load.csv": load_csv, "profile.rows": profile_rows, "fix.text": fix_text,
    "fix.numbers": fix_numbers, "merge.fixes": merge_fixes,
    "final.check": final_check, "write.out": write_out,
})

plan = compile_route(bench, {s.id: s.candidates[0] for s in bench.leaf_stages})
run = execute.run(plan, runtime, workspace=WORK)
print(run.text())

plan plan:ab1883224457d35ccdfb9…
7 steps in 0.002s — ok
  ok   load             0.000s  load.csv
  ok   profile          0.000s  profile.rows
  ok   text             0.000s  fix.text
  ok   numbers          0.000s  fix.numbers
  ok   merge            0.000s  merge.fixes
  ok   check            0.000s  final.check
  ok   write            0.000s  write.out  [file.write]
  file /home/username/code_projects/repos/browsergraph/notebooks/work/clean.csv  434 bytes  sha256:a3abdaaf5…
  file /home/username/code_projects/repos/browsergraph/notebooks/work/cleaning-report.json  997 bytes  sha256:099dc6143…


## Before

In [6]:
before = run.output("profile")
for key, value in before.items():
    print(f"  {key:<20} {value}")

  rows                 12
  duplicate_rows       2
  split_by_comma       2
  padded_values        5
  blank_values         1
  city_spellings       ['Bristol', 'Leeds', 'London', 'london']
  category_spellings   ['RETAIL', 'Retail', 'Wholesale', 'retail', 'wholesale']


## After

In [7]:
report = run.values[("check", "report")]
rows = run.values[("check", "rows")]

print(f"{before['rows']} rows in, {len(rows)} rows out\n")
print("what changed:")
for note in report["changes"]:
    print(f"  id {note['id']:<3} {note['action']:<9} {note['why']}")

print(f"\n{'id':<4}{'city':<10}{'category':<12}{'amount':>12}   flag")
for row in rows:
    amount = f"{row['amount']:,.2f}" if row["amount"] is not None else "—"
    print(f"{row['id']:<4}{row['city']:<10}{row['category']:<12}{amount:>12}   {row['flag']}")

12 rows in, 10 rows out

what changed:
  id 9   repaired  amount was split by its own comma
  id 9   repaired  amount was split by its own comma
  id 2   dropped   duplicate row
  id 5   flagged   no amount
  id 7   flagged   amount 99,999,999 is far above the rest
  id 9   dropped   duplicate row

id  city      category          amount   flag
1   London    Retail          1,200.00   
2   Leeds     Retail            980.50   
3   Leeds     Retail          1,050.00   
4   Bristol   Wholesale         640.00   
5   London    Retail                 —   missing amount
6   Bristol   Wholesale         720.25   
7   Leeds     Retail      99,999,999.00   suspicious amount
8   Bristol   Wholesale         540.10   
9   London    Retail          1,340.00   
10  Leeds     Wholesale         610.00   


Note what the cleaner did **not** do. The huge amount is flagged, not deleted.
It might be real. Silently dropping it would change someone's totals with no
trace.

In [8]:
print("files written:")
for art in run.artifacts:
    print(f"  {art.path:<34} {art.bytes:>8,} bytes  {art.digest[:18]}…")

files written:
  /home/username/code_projects/repos/browsergraph/notebooks/work/clean.csv      434 bytes  sha256:a3abdaaf552…
  /home/username/code_projects/repos/browsergraph/notebooks/work/cleaning-report.json      997 bytes  sha256:099dc61433f…
